# Session 12: Classification & ensembles

Practice companion for New. **Synthetic demonstration data**, not the lecturer's original dataset. Run the code cells top to bottom. The original slide snippets remain available separately in the website.

Requires numpy, pandas, matplotlib, seaborn, scikit-learn, and plotly. Install missing packages with `%pip install numpy pandas matplotlib seaborn scikit-learn plotly`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
X, y = make_classification(n_samples=500, n_features=6, n_informative=4,
                           weights=[0.8, 0.2], random_state=42)
X = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(6)])
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=.2, random_state=42)
models = {
    'Logistic': make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000)),
    'Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
}
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    print(name, 'ROC AUC:', roc_auc_score(y_test, proba), 'Average precision:', average_precision_score(y_test, proba))
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds, zero_division=0))
# Comparison is educational. Use CV for model selection, then a final untouched test.


## Your turn
Fit a logistic baseline and a Random Forest, evaluate them on the same held-out split, and explain one prediction. Keep threshold and model selection separate from the final test.

## Watch out
The slide's waterfall snippet passes legacy SHAP arrays; modern waterfall plots need an Explanation for one sample and one output. The practice notebook demonstrates this explicitly.

## Optional: XGBoost and SHAP
Install the optional packages before running this extension.

In [ ]:
# Optional extensions: install shap and xgboost in your notebook environment first.
# %pip install shap xgboost
import shap
from xgboost import XGBClassifier
xgb = XGBClassifier(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
print('XGBoost ROC AUC:', roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1]))
rf = models['Forest']
explanation = shap.TreeExplainer(rf)(X_test)
# For binary sklearn forests, SHAP may return (samples, features, outputs).
positive = explanation[:, :, 1] if explanation.values.ndim == 3 else explanation
shap.plots.beeswarm(positive)
shap.plots.waterfall(positive[0])
